# ASCON-AEAD-128 Auto Tight-Trigger Key-Byte Trace Collection

This notebook is adapted from the ACORNv3 auto collection workflow.

Goal:

```text
Collect byte-wise tight-trigger random-key traces for ASCON-AEAD-128
TARGET_BYTE = 0 to 15
30,000 valid traces per byte
Chunked saving to survive interruption
Metadata labels compatible with Top-k key-byte recovery training
```

Recommended location:

```text
C:\Users\thetp\Downloads\ASCON_FYP\02_Notebooks\01_collection
```

Important firmware assumption:

This notebook expects an ASCON tight-trigger firmware with SimpleSerial commands:

```text
k command: receives 16-byte ASCON key
n command: receives 16-byte ASCON nonce
p command: receives 1-byte target byte from 0 to 15 and performs selected-byte trigger capture
r response: returns 1-byte checksum/status
```

If your firmware does not support the `n` command, set `USE_NONCE_COMMAND = False`.

In [ ]:
# Cell 1 - Configuration

from pathlib import Path
from datetime import datetime
import time, json, gc, os, re, shutil, subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_DIR = Path(r"C:\Users\thetp\Downloads\ASCON_FYP")

# Change this to your actual ASCON firmware folder.
FW_DIR = PROJECT_DIR / "01_Firmware" / "tight_trigger_keybyte" / "simpleserial-ascon-keyabsorb"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OUT_ROOT = PROJECT_DIR / "03_Data" / "tight_trigger_keybyte_auto30k"
OUT_DIR = OUT_ROOT / f"ascon_aead128_tighttrigger_30k_per_byte_{RUN_ID}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLATFORM = "CWLITEXMEGA"
SS_VER = "SS_VER_1_1"

KEY_LEN = 16
NONCE_LEN = 16
TARGET_BYTE_LEN = 1
RESP_LEN = 1
TRACE_LEN = 2000

# If your ASCON firmware has simpleserial_addcmd('n', 16, get_nonce), keep this True.
# If nonce is fixed inside firmware, set this False.
USE_NONCE_COMMAND = True

# For initial testing, keep TEST_MODE = True.
# After sanity check passes, set TEST_MODE = False.
TEST_MODE = True

N_TRACES_PER_BYTE_TEST = 100
N_TRACES_PER_BYTE_FULL = 30_000

CHUNK_SIZE_TEST = 100
CHUNK_SIZE_FULL = 1000

# First test byte 0 and 1 only, then switch to list(range(16)).
TARGET_BYTES_TEST = [0, 1]
TARGET_BYTES_FULL = list(range(16))

if TEST_MODE:
    N_TRACES_PER_BYTE = N_TRACES_PER_BYTE_TEST
    CHUNK_SIZE = CHUNK_SIZE_TEST
    TARGET_BYTES = TARGET_BYTES_TEST
else:
    N_TRACES_PER_BYTE = N_TRACES_PER_BYTE_FULL
    CHUNK_SIZE = CHUNK_SIZE_FULL
    TARGET_BYTES = TARGET_BYTES_FULL

# Optional fixed values for cleaner experiments.
# For random-key profiled training, key should be random.
# Nonce can be fixed or random. Start with fixed nonce for a cleaner key-byte leakage dataset.
FIXED_NONCE = bytes.fromhex("00000000000000000000000000000000")
RANDOM_NONCE_PER_TRACE = False

print("PROJECT_DIR:", PROJECT_DIR)
print("FW_DIR:", FW_DIR)
print("FW_DIR exists:", FW_DIR.exists())
print("OUT_DIR:", OUT_DIR)
print("TEST_MODE:", TEST_MODE)
print("N_TRACES_PER_BYTE:", N_TRACES_PER_BYTE)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("TARGET_BYTES:", TARGET_BYTES)
print("USE_NONCE_COMMAND:", USE_NONCE_COMMAND)
print("RANDOM_NONCE_PER_TRACE:", RANDOM_NONCE_PER_TRACE)

In [ ]:
# Cell 2 - Inspect ASCON tight-trigger firmware

if not FW_DIR.exists():
    raise FileNotFoundError(f"Firmware folder not found: {FW_DIR}")

print("Files in FW_DIR:")
for p in sorted(FW_DIR.iterdir()):
    print(" -", p.name)

c_files = list(FW_DIR.glob("*.c"))
if not c_files:
    raise FileNotFoundError(f"No .c file found in {FW_DIR}")

print("\nSimpleSerial command and trigger inspection:")
for cfile in c_files:
    text = cfile.read_text(errors="ignore").splitlines()
    print("\n===", cfile.name, "===")
    for i, line in enumerate(text, start=1):
        if (
            "simpleserial_addcmd" in line
            or "simpleserial_put" in line
            or "trigger_high" in line
            or "trigger_low" in line
        ):
            print(f"{i:04d}: {line}")

print("\nExpected firmware interface:")
print(" - k command accepts 16-byte key")
print(" - n command accepts 16-byte nonce, unless USE_NONCE_COMMAND=False")
print(" - p command accepts 1-byte target key byte")
print(" - trigger_high/trigger_low surrounds selected ASCON key-byte operation")
print(" - r response returns 1 byte")

In [ ]:
# Cell 3 - Patch Makefile if needed and build firmware

makefile_path = FW_DIR / "Makefile"
if not makefile_path.exists():
    raise FileNotFoundError(f"Makefile not found: {makefile_path}")

make_text = makefile_path.read_text(errors="ignore")

backup_path = FW_DIR / "Makefile.original_backup"
if not backup_path.exists():
    backup_path.write_text(make_text, encoding="utf-8")
    print("Backed up original Makefile to:", backup_path)

# Ensure ChipWhisperer build system variable exists.
if "FIRMWAREPATH" not in make_text:
    print("Patching Makefile with FIRMWAREPATH...")
    patched = make_text.replace(
        "include $(CHIPWHISPERER_DIR)/firmware/mcu/Makefile.inc",
        "FIRMWAREPATH = C:/Users/thetp/ChipWhisperer/chipwhisperer/firmware/mcu\ninclude $(FIRMWAREPATH)/Makefile.inc"
    )

    if patched == make_text:
        lines = []
        for line in make_text.splitlines():
            if "Makefile.inc" not in line:
                lines.append(line)
        lines.append("")
        lines.append("FIRMWAREPATH = C:/Users/thetp/ChipWhisperer/chipwhisperer/firmware/mcu")
        lines.append("include $(FIRMWAREPATH)/Makefile.inc")
        patched = "\n".join(lines) + "\n"

    makefile_path.write_text(patched, encoding="utf-8")
    print("Makefile patched.")
else:
    print("Makefile already contains FIRMWAREPATH.")

# Build firmware.
print("\nBuilding firmware...")
result = subprocess.run(
    ["make", "PLATFORM=" + PLATFORM, "CRYPTO_TARGET=NONE", "SS_VER=" + SS_VER],
    cwd=str(FW_DIR),
    capture_output=True,
    text=True,
    shell=True,
)

print("Return code:", result.returncode)
print("STDOUT:\n", result.stdout[-3000:])
print("STDERR:\n", result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError("Firmware build failed.")

hex_files = sorted(FW_DIR.glob("*.hex"))
if not hex_files:
    raise FileNotFoundError("No .hex file found after build.")

HEX_PATH = hex_files[0]
print("HEX_PATH:", HEX_PATH)

In [ ]:
# Cell 4 - Connect and flash ASCON tight-trigger firmware

import chipwhisperer as cw

for obj_name in ["target", "scope"]:
    if obj_name in globals():
        try:
            globals()[obj_name].dis()
        except Exception:
            pass

time.sleep(1)

scope = cw.scope()
target = cw.target(scope)
scope.default_setup()

scope.adc.samples = TRACE_LEN
scope.adc.timeout = 5

print("Connected.")
print("Flashing:", HEX_PATH)
cw.program_target(scope, cw.programmers.XMEGAProgrammer, str(HEX_PATH))

time.sleep(1)
try:
    target.flush()
except Exception:
    pass

print("Flashed ASCON-AEAD-128 tight-trigger key-byte firmware.")
print("ADC samples:", scope.adc.samples)
print("ADC timeout:", scope.adc.timeout)

In [ ]:
# Cell 5 - Helper functions for ASCON tight-trigger key-byte capture

import time
import re
import numpy as np

def flush_target(delay=0.01):
    try:
        target.flush()
    except Exception:
        pass
    time.sleep(delay)

def random_bytes(n):
    return bytes(np.random.randint(0, 256, size=n, dtype=np.uint8).tolist())

def parse_key_byte(key_bytes, target_byte):
    return int(key_bytes[target_byte])

def byte_bits_lsb(byte_val):
    return [(int(byte_val) >> b) & 1 for b in range(8)]

def set_key(key_bytes, delay=0.015):
    assert len(key_bytes) == KEY_LEN
    flush_target(0.003)
    target.simpleserial_write('k', bytearray(key_bytes))
    time.sleep(delay)
    try:
        _ = target.read(timeout=0.03)
    except Exception:
        pass

def set_nonce(nonce_bytes, delay=0.010):
    assert len(nonce_bytes) == NONCE_LEN
    if not USE_NONCE_COMMAND:
        return
    flush_target(0.003)
    target.simpleserial_write('n', bytearray(nonce_bytes))
    time.sleep(delay)
    try:
        _ = target.read(timeout=0.03)
    except Exception:
        pass

def read_tight_response_raw(timeout_s=0.4, debug=False):
    chunks = []
    start = time.time()
    pattern = r"r([0-9a-fA-F]{" + str(RESP_LEN * 2) + r"})"

    while time.time() - start < timeout_s:
        try:
            s = target.read(timeout=0.02)
            if s:
                chunks.append(str(s))
                joined = "".join(chunks)
                m = re.search(pattern, joined)
                if m:
                    return bytes.fromhex(m.group(1)), joined
        except Exception:
            pass
        time.sleep(0.003)

    joined = "".join(chunks)
    if debug:
        print("Raw joined:", repr(joined))
    return None, joined

def capture_ascon_keybyte_trace(target_byte, key_bytes=None, nonce_bytes=None, timeout=0.4, debug=False):
    assert 0 <= int(target_byte) <= 15

    if key_bytes is None:
        key_bytes = random_bytes(KEY_LEN)
    assert len(key_bytes) == KEY_LEN

    if nonce_bytes is None:
        nonce_bytes = random_bytes(NONCE_LEN) if RANDOM_NONCE_PER_TRACE else FIXED_NONCE
    assert len(nonce_bytes) == NONCE_LEN

    key_byte_value = parse_key_byte(key_bytes, target_byte)
    bits = byte_bits_lsb(key_byte_value)

    set_key(key_bytes, delay=0.015)
    set_nonce(nonce_bytes, delay=0.010)
    flush_target(0.002)

    scope.arm()

    # p sends the selected target byte.
    # Firmware should trigger only around the selected key-byte operation.
    target.simpleserial_write('p', bytearray([int(target_byte)]))
    ret = scope.capture()

    out, raw_text = read_tight_response_raw(timeout_s=timeout, debug=debug)

    wave = scope.get_last_trace()
    if wave is not None:
        wave = np.asarray(wave, dtype=np.float32)

    trig = int(getattr(scope.adc, "trig_count", 0))

    return {
        "wave": wave,
        "out": out,
        "raw_text": raw_text,
        "trig": trig,
        "ret": ret,
        "key_bytes": key_bytes,
        "nonce_bytes": nonce_bytes,
        "key_byte_value": key_byte_value,
        "bits": bits,
    }

In [ ]:
# Cell 6 - Sanity check for ASCON TARGET_BYTE 0 and 1

for tb in TARGET_BYTES:
    result = capture_ascon_keybyte_trace(tb, timeout=1.0, debug=True)
    wave = result["wave"]
    out = result["out"]
    trig = result["trig"]

    print("\nTARGET_BYTE:", tb)
    print("output:", None if out is None else out.hex())
    print("trigger count:", trig)
    print("trace length:", None if wave is None else len(wave))
    print("key byte value:", result["key_byte_value"])
    print("bits LSB:", result["bits"])

    if wave is None:
        raise RuntimeError("No trace captured.")
    if trig == 0:
        raise RuntimeError("Trigger count is 0.")
    if len(wave) != TRACE_LEN:
        print("WARNING: trace length != TRACE_LEN", len(wave), TRACE_LEN)

print("\nSUCCESS: ASCON tight-trigger sanity check passed.")

In [ ]:
# Cell 7 - Save run configuration

config = {
    "algorithm": "ASCON-AEAD-128",
    "project_dir": str(PROJECT_DIR),
    "firmware_dir": str(FW_DIR),
    "hex_path": str(HEX_PATH),
    "out_dir": str(OUT_DIR),
    "platform": PLATFORM,
    "ss_ver": SS_VER,
    "test_mode": TEST_MODE,
    "n_traces_per_byte": N_TRACES_PER_BYTE,
    "chunk_size": CHUNK_SIZE,
    "target_bytes": TARGET_BYTES,
    "trace_len": TRACE_LEN,
    "key_len": KEY_LEN,
    "nonce_len": NONCE_LEN,
    "target_byte_len": TARGET_BYTE_LEN,
    "resp_len": RESP_LEN,
    "use_nonce_command": USE_NONCE_COMMAND,
    "random_nonce_per_trace": RANDOM_NONCE_PER_TRACE,
    "fixed_nonce_hex": FIXED_NONCE.hex(),
    "created_at": datetime.now().isoformat(),
    "method": "random key per trace, selectable target byte 0..15, tight-trigger ASCON-AEAD-128 key-byte operation",
}

config_path = OUT_DIR / "collection_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("Saved config:", config_path)
print(json.dumps(config, indent=2))

In [ ]:
# Cell 8 - Collection functions

def save_byte_chunk(target_byte, chunk_index, traces, metadata_rows):
    byte_dir = OUT_DIR / f"byte_{target_byte:02d}"
    byte_dir.mkdir(parents=True, exist_ok=True)

    traces_arr = np.asarray(traces, dtype=np.float32)

    npz_path = byte_dir / f"byte{target_byte:02d}_chunk_{chunk_index:04d}_traces.npz"
    csv_path = byte_dir / f"byte{target_byte:02d}_chunk_{chunk_index:04d}_metadata.csv"

    if npz_path.exists() or csv_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing chunk: {npz_path} or {csv_path}")

    np.savez_compressed(npz_path, traces=traces_arr)
    pd.DataFrame(metadata_rows).to_csv(csv_path, index=False)

    print(f"Saved ASCON byte {target_byte:02d} chunk {chunk_index:04d}: {traces_arr.shape}")

def collect_target_byte(target_byte, n_traces):
    byte_dir = OUT_DIR / f"byte_{target_byte:02d}"
    byte_dir.mkdir(parents=True, exist_ok=True)

    traces = []
    metadata_rows = []
    errors = []
    chunk_index = 0
    collected = 0
    attempts = 0
    max_attempts = n_traces * 3

    pbar = tqdm(total=n_traces, desc=f"ASCON TARGET_BYTE {target_byte:02d}")

    while collected < n_traces and attempts < max_attempts:
        attempts += 1

        try:
            key_bytes = random_bytes(KEY_LEN)
            nonce_bytes = random_bytes(NONCE_LEN) if RANDOM_NONCE_PER_TRACE else FIXED_NONCE

            result = capture_ascon_keybyte_trace(
                target_byte=target_byte,
                key_bytes=key_bytes,
                nonce_bytes=nonce_bytes,
                timeout=0.4,
                debug=False,
            )

            wave = result["wave"]
            out = result["out"]
            trig = result["trig"]
            ret = result["ret"]

            ok = (
                wave is not None
                and trig > 0
                and len(wave) == TRACE_LEN
            )

            output_hex = None if out is None else out.hex()

            if not ok:
                errors.append({
                    "target_byte": target_byte,
                    "attempt_index": attempts,
                    "valid_trace_index": collected,
                    "error": "invalid_capture",
                    "ret": str(ret),
                    "output_hex": output_hex,
                    "trigger_count": trig,
                    "trace_len": None if wave is None else len(wave),
                    "key_hex": key_bytes.hex(),
                    "nonce_hex": nonce_bytes.hex(),
                })
                continue

            key_byte_value = result["key_byte_value"]
            bits = result["bits"]

            row = {
                "algorithm": "ASCON-AEAD-128",
                "target_byte": target_byte,
                "trace_index": collected,
                "attempt_index": attempts,
                "key_hex": key_bytes.hex(),
                "nonce_hex": nonce_bytes.hex(),
                "key_byte_value": key_byte_value,
                "key_byte_hex": f"{key_byte_value:02x}",
                "key_byte_hw": int(sum(bits)),
                "output_hex": output_hex,
                "trigger_count": trig,
                "capture_ret": ret,
            }

            for b in range(8):
                row[f"bit{b}_lsb"] = int(bits[b])

            metadata_rows.append(row)
            traces.append(wave)

            collected += 1
            pbar.update(1)

            if len(traces) >= CHUNK_SIZE:
                save_byte_chunk(target_byte, chunk_index, traces, metadata_rows)
                chunk_index += 1
                traces = []
                metadata_rows = []
                gc.collect()

        except KeyboardInterrupt:
            print("\nKeyboardInterrupt received. Saving current partial chunk...")
            break

        except Exception as e:
            errors.append({
                "target_byte": target_byte,
                "attempt_index": attempts,
                "valid_trace_index": collected,
                "error": repr(e),
            })
            continue

    pbar.close()

    if traces:
        save_byte_chunk(target_byte, chunk_index, traces, metadata_rows)

    errors_path = byte_dir / f"byte{target_byte:02d}_collection_errors.json"
    errors_path.write_text(json.dumps(errors, indent=2), encoding="utf-8")

    summary = {
        "algorithm": "ASCON-AEAD-128",
        "target_byte": target_byte,
        "requested": n_traces,
        "collected": collected,
        "attempts": attempts,
        "errors": len(errors),
        "byte_dir": str(byte_dir),
        "finished_at": datetime.now().isoformat(),
    }

    summary_path = byte_dir / f"byte{target_byte:02d}_collection_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"\nFinished ASCON TARGET_BYTE {target_byte:02d}")
    print(json.dumps(summary, indent=2))

    return summary

In [ ]:
# Cell 9 - Run automated ASCON collection

all_summaries = []

for target_byte in TARGET_BYTES:
    print("\n" + "="*80)
    print(f"STARTING ASCON TARGET_BYTE {target_byte}")
    print("="*80)

    summary = collect_target_byte(target_byte, N_TRACES_PER_BYTE)
    all_summaries.append(summary)

    summary_path = OUT_DIR / "all_bytes_collection_summary_so_far.json"
    summary_path.write_text(json.dumps(all_summaries, indent=2), encoding="utf-8")

print("\nALL REQUESTED ASCON TARGET BYTES FINISHED.")

final_summary = {
    "algorithm": "ASCON-AEAD-128",
    "n_traces_per_byte": N_TRACES_PER_BYTE,
    "target_bytes": TARGET_BYTES,
    "total_requested": int(N_TRACES_PER_BYTE * len(TARGET_BYTES)),
    "total_collected": int(sum(s["collected"] for s in all_summaries)),
    "total_errors": int(sum(s["errors"] for s in all_summaries)),
    "out_dir": str(OUT_DIR),
    "finished_at": datetime.now().isoformat(),
    "summaries": all_summaries,
}

final_summary_path = OUT_DIR / "final_collection_summary.json"
final_summary_path.write_text(json.dumps(final_summary, indent=2), encoding="utf-8")

print(json.dumps(final_summary, indent=2))
print("Saved final summary:", final_summary_path)

In [ ]:
# Cell 10 - Verify saved ASCON dataset structure

rows = []

for byte_dir in sorted(OUT_DIR.glob("byte_*")):
    if not byte_dir.is_dir():
        continue

    target_byte = int(byte_dir.name.split("_")[1])
    trace_files = sorted(byte_dir.glob("*_traces.npz"))
    meta_files = sorted(byte_dir.glob("*_metadata.csv"))

    n_trace_rows = 0
    shapes = []

    for tf in trace_files:
        try:
            arr = np.load(tf)["traces"]
            shapes.append(tuple(arr.shape))
            n_trace_rows += arr.shape[0]
        except Exception as e:
            shapes.append(("ERROR", str(e)))

    n_meta_rows = 0
    metadata_cols = None

    for mf in meta_files:
        try:
            df = pd.read_csv(mf)
            n_meta_rows += len(df)
            if metadata_cols is None:
                metadata_cols = list(df.columns)
        except Exception:
            pass

    rows.append({
        "target_byte": target_byte,
        "trace_files": len(trace_files),
        "metadata_files": len(meta_files),
        "trace_rows": n_trace_rows,
        "metadata_rows": n_meta_rows,
        "first_shape": shapes[0] if shapes else None,
        "last_shape": shapes[-1] if shapes else None,
        "metadata_cols": metadata_cols,
    })

verify_df = pd.DataFrame(rows).sort_values("target_byte")
verify_path = OUT_DIR / "dataset_verification_summary.csv"
verify_df.to_csv(verify_path, index=False)

print("Saved verification summary:", verify_path)
display(verify_df)

expected_chunks = int(np.ceil(N_TRACES_PER_BYTE / CHUNK_SIZE))
print("\nExpected chunks per byte:", expected_chunks)
print("Expected traces per byte:", N_TRACES_PER_BYTE)